---
format:
  html:
    code-fold: false
jupyter: python3
---

# Jupyter Activity: p-ary Expansions

Math 140H, Honors Calculus I

## Before we start: how to use a Jupyter notebook

This is the first notebook of the semester, so we start a quick orientation:

- A notebook is a sequence of **cells**. This text lives in a *markdown cell* (formatted text);
  cells with `code` in the toolbar are *code cells* (Python).
- Click a cell and press **Shift+Enter** to run it and move to the next one. Code cells must be run
  in order the first time through, top to bottom — a later cell may use a function or variable that
  an earlier cell defined.
- If something breaks, it is often because a cell was skipped or run twice out of order. Use the
  menu command *Run All* (or *Restart Kernel and Run All*) to run the whole notebook cleanly from
  the top.
- You do **not** need to know Python already. Every function below is commented line by line —
  read the comments (the lines starting with `#`) as a description of the math, and the code as
  one particular way of writing it down.

You can think of a code cell very much like a fancy calculator. Move to the next cell an run it.

In [21]:
16+(53*8.7777-128)

353.2181

Careful: In Python, exponentiation is `**`, not `^`. 

In [23]:
2**4

16

In [24]:
2^4

6

`^` in Python is the operation called `bitwise XOR`; that explains the strange result above. 

We will learn more about Jupyter notebooks (and a little bit more about Python) im the course of this semester. Today, we will simply experiment a bit with pre-defined Python function that converts rational numbers to $p$-ary expansions. Go ahead and run the cells below one after the other (do not forget to read the text inbetween.)

---

## Importing Python modules

A lot functionality in Python is provided by *modules*. The following cell loads the modules we need for this notebook.

In [27]:
# We import three tools before doing anything else:
#
#   math     -- the standard library module with math.floor(y), the greatest integer <= y.
#               (We need floor of *exact fractions*, not of floating-point approximations --
#               see the note below.)
#   Fraction -- from the 'fractions' module. A Fraction stores a rational number as an exact
#               numerator/denominator pair, e.g. Fraction(1, 3) is *exactly* one third, forever.
#               This matters a lot here: a computer's ordinary decimal numbers (called 'floats')
#               are themselves binary approximations, so 1/3 as a float is already slightly wrong,
#               and the tiny error would snowball after enough steps of the algorithm below.
#               Using Fraction means every digit we compute is provably correct, not just close.
#   numpy    -- imported as 'np' by convention. We use it only to store the list of digits we
#               compute as a numpy array, which is a convenient container for a sequence of numbers.
import math
from fractions import Fraction
import numpy as np

The cells below show the advantage of the `Fraction` function: 

In [31]:
1/2+1/3   # Computer will use decimal expansions

0.8333333333333333

In [33]:
Fraction(1,2)+Fraction(1,3)   # will preserve the precise fraction representation

Fraction(5, 6)

## From a rational number to its *p*-ary expansion

We generalize the nested interval construction to base $p$: given $x$, set
$$
c_0 := \lfloor x \rfloor,
$$
and then repeatedly zoom in. Having peeled off $c_0,c_1,\dots,c_{n-1}$, look at what is left over,

$$
r_{n-1} \;=\; x - c_0 - \frac{1}{p}c_1 - \cdots - \frac{1}{p^{n-1}}c_{n-1} \;\in [0, p^{-(n-1)}),
$$

multiply it by $p^{n-1}$ so it lands back in $[0,1)$, and read off the next digit as the integer
part of $p$ times that:

$$
c_n = \lfloor p \cdot (p^{n-1} r_{n-1}) \rfloor.
$$
(Multiplying by $p$ here means zooming in by a factor of $p$.)

The function below does exactly this, but it tracks the *rescaled* remainder (called `remainder`
in the code, always kept in $[0,1)$) rather than $r_{n-1}$ itself, since that is what you actually
multiply by $p$ at each step:

1. `remainder = x - floor(x)`, the fractional part of $x$, in $[0,1)$.
2. Repeat: multiply `remainder` by `p`. Its integer part is the next digit; subtract that integer
   part off, leaving a new `remainder` back in $[0,1)$, and go again.


In [38]:
def rational_to_p_ary(x, p, max_digits=50):
    """
    Compute the base-p expansion of a rational number x.

    Parameters
    ----------
    x : Fraction, int, or a string like "7/12"
        The rational number to expand. (Do NOT pass a plain float such as 0.76 -- floats are
        already binary approximations, so we would be expanding the wrong number. Use
        Fraction(76, 100) instead. See the discussion above.)
    p : int
        The base, an integer > 1 (p = 2 for binary, p = 10 for the usual decimal expansion, ...).
    max_digits : int, optional (default = 50)
        how many fractional digits to compute 
        
    Returns
    -------
    integer_part : int
        c_0, the integer part of x (i.e. floor(x)).
    digits : numpy array of int
        The fractional digits c_1, c_2, c_3, ... that were computed, in order.
    repeat_start : int or None
        If the expansion is eventually periodic, the index (0-based, into `digits`) where the
        repeating block begins. If the expansion terminates exactly (all later digits are 0),
        this is None.
    """
    if not isinstance(p, int) or p <= 1:
        raise ValueError("the base p must be an integer greater than 1")

    # Convert x to an exact fraction, no matter what form it was given in.
    x = Fraction(x)

    # Step 1: peel off the integer part c_0 = floor(x). What remains, x - c_0, is the
    # 'fractional part' of x and always lies in [0, 1).
    integer_part = math.floor(x)
    remainder = x - integer_part

    digits = []            # the digits c_1, c_2, ... found so far, in order
    seen_remainders = {}   # remembers, for each remainder value we've seen, which step it
                            # first showed up at -- this is how we notice a repeat
    repeat_start = None

    for step in range(max_digits):
        # Zooming into the next sub-interval of
        # length 1/p means multiplying the remainder by p; the integer part of the result is
        # exactly which of the p equal pieces x fell into, i.e. the next digit.
        remainder = remainder * p
        next_digit = math.floor(remainder)
        digits.append(next_digit)

        # Subtract off the digit we just read, leaving a fresh remainder back in [0, 1),
        # ready for the next iteration.
        remainder = remainder - next_digit

    return integer_part, np.array(digits, dtype=int), repeat_start

The cell below defines a small helper function to print an expansion in a readable form. We deliberately write it as
`c_0 + 0.c_1c_2c_3...` with an explicit `+`, matching the lecture notes' warning: if $c_0$ is
negative, the fractional digits are still *added* to it, not subtracted, so `-3.25` written the
ordinary way would be dangerously ambiguous. Writing `-3 + 0.25` instead leaves no room for that
mistake.

In [16]:
def format_expansion(integer_part, digits, p, repeat_start=None):
    """
    Turn the output of rational_to_p_ary into a readable string, e.g.
        format_expansion(0, np.array([1, 1, 0]), 2, None) -> '0 + 0.110  (base 2)'
        format_expansion(0, np.array([3]), 10, 0)          -> '0 + 0.(3)  (base 10)'   # = 0.333...
    """
    # Turn each digit into a one-character string. In any base p <= 10 these are ordinary digits;
    # for p > 10 we would need letters (like hexadecimal's a-f), which we don't handle here.
    digit_chars = [str(int(d)) for d in digits]

    if repeat_start is None:
        # No repeating block: just list every digit we have.
        fractional_part = "".join(digit_chars) if len(digit_chars) > 0 else "0"
    else:
        # Split into the digits before the repeat (head) and the repeating block itself (tail),
        # and wrap the tail in parentheses to mark it as repeating forever.
        head = "".join(digit_chars[:repeat_start])
        tail = "".join(digit_chars[repeat_start:])
        fractional_part = f"{head}({tail})" if head else f"({tail})"

    return f"{integer_part} + 0.{fractional_part}  (base {p})"

## Checking the examples
In the lecture notes, we found the first three *binary* digits of $x = 0.76$ by hand:
$c_1=1,\ c_2=1,\ c_3=0$, i.e. $0.76 = 0.110\ldots$ in base 2. Let's confirm that.

In [ ]:
integer_part, digits, repeat_start = rational_to_p_ary(Fraction(76, 100), 2, max_digits=30)
print(format_expansion(integer_part, digits, 2, repeat_start))
print("first three fractional digits:", digits[:3], " -- matches c1, c2, c3 = 1, 1, 0 from Example")

0 + 0.110000101000111101011100001010  (base 2)
first three fractional digits: [1 1 0]  -- matches c1, c2, c3 = 1, 1, 0 from Example 2.6


Now find the first three fractional digits of the base-2
expansion of $x = 0.3$, and of the base-3 expansion of $x = 0.76$. Work those out on paper first --
then run the cell below to check yourself.

In [44]:
ip, d, r = rational_to_p_ary(Fraction(3, 10), 2, max_digits=20)
print("0.3 in base 2:", format_expansion(ip, d, 2, r))
print("first three digits:", d[:3])
print()
ip, d, r = rational_to_p_ary(Fraction(76, 100), 3, max_digits=20)
print("0.76 in base 3:", format_expansion(ip, d, 3, r))
print("first three digits:", d[:3])

0.3 in base 2: 0 + 0.01001100110011001100  (base 2)
first three digits: [0 1 0]

0.76 in base 3: 0 + 0.20211200100201102212  (base 3)
first three digits: [2 0 2]


## A few more instructive examples

- $1/3$ in base 10 has *no* terminating part at all -- the repeating block starts immediately.
- $1/6$ in base 10 has one non-repeating digit followed by a repeating block (matches
  $1/6 = 0.1\overline{6}$).
- $1/10$ terminates in base 10 (as expected, $1/10 = 0.1$ exactly) but is *purely repeating* in
  base 2 -- this is the mathematical reason `0.1` cannot be stored exactly as an ordinary
  floating-point number, and is why we insisted on `Fraction` above instead of Python floats.
- Negative $x$: $x = -11/4 = -2.75$. Its integer part is $c_0 = \lfloor -2.75\rfloor = -3$, and the
  digits describe $0.25$, which gets *added* to $-3$ (giving back $-2.75$), not subtracted.

In [46]:
for label, value, base, n in [
    ("1/3", Fraction(1, 3), 10, 10),
    ("1/6", Fraction(1, 6), 10, 10),
    ("1/10", Fraction(1, 10), 10, 10),
    ("1/10", Fraction(1, 10), 2, 30),
    ("-11/4", Fraction(-11, 4), 10, 10),
]:
    ip, d, r = rational_to_p_ary(value, base, max_digits=n)
    print(f"{label:>6} = {format_expansion(ip, d, base, r)}")

   1/3 = 0 + 0.3333333333  (base 10)
   1/6 = 0 + 0.1666666666  (base 10)
  1/10 = 0 + 0.1000000000  (base 10)
  1/10 = 0 + 0.000110011001100110011001100110  (base 2)
 -11/4 = -3 + 0.2500000000  (base 10)


## Try it yourself

Pick your own rational number $x$ and a base $p > 1$. Use `rational_to_p_ary` to get its
$p$-ary expansion.

In [48]:
# Replace the values below with your own choice of x (as a Fraction), p.
x = Fraction(1, 2)
p = 10


ip, d, r = rational_to_p_ary(x, p, max_digits=30)
print(f"x in base {p}:", format_expansion(ip, d, p, r))

x in base 10: 0 + 0.500000000000000000000000000000  (base 10)


## Food for thought

**Question 1:** For a fixed base $p$, which fractions will have a $p$-ary expansion that will eventually be identically $0$? (such as $1/2$ in base 10) Try to formulate a hypothesis and test it using the cell above.

**Question 2:** Is it true that all $p$-ary expansions of rational numbers eventually become periodic? If yes, why is this the case?